In [100]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage  # ← ADD THIS
import requests

@tool
def get_exchange_rate(from_currency: str, to_currency: str) -> float:
    """Get currency exchange rate"""
    url = f"https://api.frankfurter.app/latest?from={from_currency}&to={to_currency}"
    response = requests.get(url)
    data = response.json()
    return data['rates'][to_currency]

llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([get_exchange_rate])

messages = [HumanMessage(content="Convert 100 USD to INR")]

# First call
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

# Execute tool
for tool_call in ai_message.tool_calls:
    if tool_call['name'] == "get_exchange_rate":
        rate = get_exchange_rate.invoke(tool_call['args'])
        tool_message = ToolMessage(
            content=str(rate),
            tool_call_id=tool_call['id']
        )
        messages.append(tool_message)
        conversion_rate = tool_message.content
        print(f"Rate: {conversion_rate}")

# Get final answer
final_response = llm_with_tools.invoke(messages)
print(f"Answer: {final_response.content}")

Rate: 95.69
Answer: 100 USD is equal to 9569 INR.


In [101]:
messages

[HumanMessage(content='Convert 100 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 57, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DkCXhQMJIBXfvQxHBOcJv46SI0Ewb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e6a86-ab2c-7b93-8136-f130d903e207-0', tool_calls=[{'name': 'get_exchange_rate', 'args': {'from_currency': 'USD', 'to_currency': 'INR'}, 'id': 'call_hIpGNI0Wo99JUH4OTwnxKaFT', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 22, 'total_tokens': 79, 'input_tok